Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 05 — rozpoznawanie przeuczenia i dobór regularyzacji

Materiał towarzyszący [`README.pl.md`](README.pl.md). Klasyfikacja tabularna z
cechami informacyjnymi i szumowymi (inny problem niż wykład). Zaczynasz
od zdiagnozowania przecieku danych w normalizacji, potem trenujesz model,
który się przeucza, i eksperymentalnie porównujesz regularyzacje.




In [ ]:
import copy  # noqa: F401 - uzywany w sekcji 5 (early stopping)

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")


## 1. Dane i podział (gotowe)

14 cech (4 informacyjne, 10 czysty szum), 2 cechy ze sztucznie większą
skalą. 100 treningowych, 200 walidacyjnych, 200 testowych.


In [ ]:
def make_data(n, n_informative, n_noise, seed, noise_std=0.3):
    g = torch.Generator().manual_seed(seed)
    n_features = n_informative + n_noise
    X = torch.randn(n, n_features, generator=g)
    true_w = torch.randn(n_informative, generator=g)
    logits = X[:, :n_informative] @ true_w + torch.randn(n, generator=g) * noise_std
    y = (logits > 0).float()
    return X, y.unsqueeze(-1)


N_TRAIN, N_VAL, N_TEST = 100, 200, 200
X_all, y_all = make_data(N_TRAIN + N_VAL + N_TEST, n_informative=4, n_noise=10, seed=0)
X_all[:, 0] *= 15.0  # cecha o wiekszej skali
X_all[:, 5] *= 8.0   # druga cecha o wiekszej skali

perm = torch.randperm(X_all.shape[0], generator=torch.Generator().manual_seed(1))
train_idx = perm[:N_TRAIN]
val_idx = perm[N_TRAIN : N_TRAIN + N_VAL]
test_idx = perm[N_TRAIN + N_VAL :]

X_train_raw, y_train = X_all[train_idx], y_all[train_idx]
X_val_raw, y_val = X_all[val_idx], y_all[val_idx]
X_test_raw, y_test = X_all[test_idx], y_all[test_idx]

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}, N_test={len(test_idx)}")
print(f"odchylenia std (surowe, przed normalizacja), pierwsze 6 cech: "
      f"{X_train_raw[:, :6].std(dim=0).tolist()}")


### Przeciek danych — zobacz, zanim przejdziesz dalej

Poniższy kod jest **celowo błędny**. Przeczytaj go uważnie przed
uruchomieniem kolejnej komórki.


In [ ]:
# UWAGA: ten kod liczy statystyki normalizacji na calym X_all, PRZED podzialem.
mean_leaky = X_all.mean(dim=0)
std_leaky = X_all.std(dim=0)

X_train_leaky = (X_train_raw - mean_leaky) / std_leaky
X_val_leaky = (X_val_raw - mean_leaky) / std_leaky
X_test_leaky = (X_test_raw - mean_leaky) / std_leaky

print(f"(bledne) srednia cechy 0: {mean_leaky[0].item():.4f}  "
      f"odchylenie std cechy 0: {std_leaky[0].item():.4f}")


**Zadanie (Markdown, 3–5 zdań):** na czym dokładnie polega błąd w
powyższym kodzie? Dlaczego to jest data leakage, nawet jeśli model
"dobrze działa"? Jakie dane, w jakim momencie, nie powinny być widoczne
przy liczeniu `mean_leaky`/`std_leaky`?


_(Twoja odpowiedź tutaj.)_

### Poprawka — Twoje zadanie

Statystyki normalizacji policz **wyłącznie** na `X_train_raw`, zastosuj
(bez przeliczania) do wszystkich trzech zbiorów.


In [ ]:
def normalize_correctly(X_train_raw, X_val_raw, X_test_raw):
    """Zwraca (mean, std, X_train, X_val, X_test). Statystyki TYLKO z X_train_raw."""
    # TODO:
    # 1. policz mean, std WYLACZNIE z X_train_raw (dim=0)
    # 2. znormalizuj X_train_raw, X_val_raw, X_test_raw tymi samymi
    #    statystykami (bez przeliczania osobno dla val/test!)
    # 3. zwroc (mean, std, X_train, X_val, X_test)
    raise NotImplementedError("TODO: poprawna normalizacja")


In [ ]:
mean_correct, std_correct, X_train, X_val, X_test = normalize_correctly(
    X_train_raw, X_val_raw, X_test_raw
)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

print(f"(poprawna) srednia cechy 0: {mean_correct[0].item():.4f}  "
      f"odchylenie std cechy 0: {std_correct[0].item():.4f}")
print(f"roznica srednich (bledna - poprawna), cecha 0: "
      f"{(mean_leaky[0] - mean_correct[0]).item():.4f}")
print(f"roznica odchylen std (bledna - poprawna), cecha 0: "
      f"{(std_leaky[0] - std_correct[0]).item():.4f}")


**Interpretacja.** Różnice średnich/odchyleń dla cechy 0 są wyraźnie
niezerowe — błędne i poprawne statystyki normalizacji to **nie** te same
liczby, mimo że pochodzą z tego samego źródłowego zbioru danych.


### Czy przeciek faktycznie zmienia wynik? Sprawdźmy, zamiast zakładać (gotowe)

Szybki, mały model wytrenowany na danych znormalizowanych błędnie i
poprawnie — porównanie na odpowiadających im (błędnie/poprawnie
znormalizowanych) zbiorach testowych. To jest dowód, nie założenie,
na który odpowiesz w pytaniu 2 sekcji 7.


In [ ]:
def _quick_train_eval(X_tr, y_tr, X_eval, y_eval, seed):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(14, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(500):
        m.train()
        opt.zero_grad()
        loss = loss_fn(m(X_tr), y_tr)
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = ((torch.sigmoid(m(X_eval)) > 0.5).float() == y_eval).float().mean().item()
    return acc


print("Dokladnosc testowa, kilka ziaren inicjalizacji:")
for seed in [0, 1, 2]:
    acc_leaky = _quick_train_eval(X_train_leaky, y_train, X_test_leaky, y_test, seed)
    acc_correct = _quick_train_eval(X_train, y_train, X_test, y_test, seed)
    print(f"  seed={seed}: normalizacja bledna test_acc={acc_leaky:.3f}   "
          f"normalizacja poprawna test_acc={acc_correct:.3f}")


## 2. Model bazowy i model o nadmiarowej pojemności (gotowe)


In [ ]:
def make_model(hidden_sizes, n_features=14, seed=0, dropout_p=0.0):
    torch.manual_seed(seed)
    layers = []
    in_dim = n_features
    for h in hidden_sizes:
        layers.append(nn.Linear(in_dim, h))
        layers.append(nn.ReLU())
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        in_dim = h
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers).to(device)


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
        return (preds == y).float().mean().item()


majority_class = y_train.mean().round()
baseline_val_acc = (torch.full_like(y_val, majority_class.item()) == y_val).float().mean().item()
print(f"Baseline (klasa wiekszosciowa): val_acc={baseline_val_acc:.3f}")

MODEL_SEED = 1
HIDDEN = [32, 32]
print(f"Model [32,32] parametry: "
      f"{sum(p.numel() for p in make_model(HIDDEN, seed=MODEL_SEED).parameters())}")


## 3. Pętla treningowa — Twoje zadanie


In [ ]:
def train_model(model, epochs, lr=0.001, weight_decay=0.0, use_adamw=False):
    """Zwraca (train_losses, val_losses). Caly X_train w jednym batchu
    (100 przykladow) - nie potrzeba mini-batchy tutaj.
    """
    # TODO:
    # 1. zbuduj optymalizator: torch.optim.AdamW jesli use_adamw, inaczej
    #    torch.optim.Adam, z lr i weight_decay
    # 2. loss_fn = nn.BCEWithLogitsLoss()
    # 3. w petli po epochs: model.train() -> zero_grad -> forward na calym
    #    X_train -> strata -> backward() -> step() -> zapisz strate
    # 4. model.eval() -> pod torch.no_grad() policz i zapisz strate na X_val/y_val
    raise NotImplementedError("TODO: train_model")


In [ ]:
baseline_model = make_model(HIDDEN, seed=MODEL_SEED)
baseline_tl, baseline_vl = train_model(baseline_model, epochs=1000, lr=0.001)

baseline_train_acc = accuracy(baseline_model, X_train, y_train)
baseline_val_acc_trained = accuracy(baseline_model, X_val, y_val)
print(f"Baseline [32,32] bez regularyzacji: train_acc={baseline_train_acc:.3f}  "
      f"val_acc={baseline_val_acc_trained:.3f}")
print(f"train_loss_final={baseline_tl[-1]:.4f}  "
      f"val_loss_min={min(baseline_vl):.4f}@{baseline_vl.index(min(baseline_vl))}  "
      f"val_loss_final={baseline_vl[-1]:.4f}")

plt.figure(figsize=(6, 4.5))
plt.plot(baseline_tl, label="trening")
plt.plot(baseline_vl, label="walidacja")
plt.xlabel("epoka")
plt.ylabel("BCE")
plt.title("Baseline [32,32], bez regularyzacji")
plt.legend()
plt.tight_layout()
plt.show()


**Zadanie (Markdown, 3–5 zdań):** jaka jest dokładność treningowa vs.
walidacyjna? W którym miejscu krzywej widać dowód przeuczenia — nie
"straty są różne", tylko konkretny kształt, który to pokazuje?


_(Twoja odpowiedź tutaj.)_

## 4. Weight decay i dropout — Twoje zadanie


In [ ]:
# TODO:
# 1. dla AdamW: przetestuj weight_decay z [0.0, 0.01, 0.1, 0.3] (uzyj
#    train_model(..., weight_decay=wd, use_adamw=True)), zapisz wyniki
#    W TYM WYTRENOWANY MODEL (np. w slowniku
#    wd_results[wd] = (model, tl, vl, accuracy(model, X_val, y_val)))
#    - model bedzie potrzebny w sekcji 6 do finalnej ewaluacji
# 2. dla dropout: przetestuj p z [0.0, 0.2, 0.4, 0.6] (make_model(...,
#    dropout_p=p), train_model(...) bez weight_decay), zapisz analogicznie
#    w dropout_results
# 3. wypisz dla obu: wartosc parametru, val_acc, val_loss_final
raise NotImplementedError("TODO: weight decay i dropout")


## 5. Early stopping — Twoje zadanie

Szkielet pętli jest gotowy — brakuje logiki zapamiętywania i przywracania
najlepszego stanu.


In [ ]:
def train_with_early_stopping(model, epochs, lr=0.001, patience=50, min_delta=1e-4):
    """Zwraca (model, train_losses, val_losses, best_epoch, stopped_epoch).
    Model zwracany MUSI miec przywrocony stan z najlepszej epoki.
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    best_val = float("inf")  # noqa: F841 - uzyj w TODO ponizej
    best_state = None  # noqa: F841 - uzyj w TODO ponizej
    best_epoch = -1
    epochs_since_improvement = 0  # noqa: F841 - uzyj w TODO ponizej
    stopped_epoch = epochs
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        opt.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            v = loss_fn(model(X_val), y_val).item()
        val_losses.append(v)

        # TODO:
        # 1. jesli v < best_val - min_delta: zaktualizuj best_val, zapisz
        #    GLEBOKA KOPIE stanu modelu (copy.deepcopy(model.state_dict()))
        #    jako best_state, zapisz best_epoch = epoch, wyzeruj
        #    epochs_since_improvement
        # 2. w przeciwnym razie: zwieksz epochs_since_improvement; jesli
        #    osiagnal patience: ustaw stopped_epoch = epoch i przerwij petle (break)
        raise NotImplementedError("TODO: logika best-state/patience")

    # TODO: przywroc najlepszy stan do modelu (model.load_state_dict(best_state))
    # (ten kod jest nieosiagalny, dopoki TODO powyzej rzuca wyjatek - to celowe)
    return model, train_losses, val_losses, best_epoch, stopped_epoch


In [ ]:
es_model = make_model(HIDDEN, seed=MODEL_SEED)
es_model, es_tl, es_vl, es_best_epoch, es_stopped_epoch = train_with_early_stopping(
    es_model, epochs=1000, patience=50
)

print(f"Zatrzymano na epoce {es_stopped_epoch}; najlepsza epoka = {es_best_epoch}")
print(f"Strata walidacyjna w najlepszej epoce = {es_vl[es_best_epoch]:.4f}")
print(f"Strata walidacyjna w ostatniej wytrenowanej epoce = {es_vl[-1]:.4f}")

restored_val_loss = nn.BCEWithLogitsLoss()(es_model(X_val), y_val).item()
print(f"Strata walidacyjna PO przywroceniu stanu modelu = {restored_val_loss:.4f}")
assert abs(restored_val_loss - es_vl[es_best_epoch]) < 1e-6, (
    "Model nie zostal poprawnie przywrocony do najlepszego stanu!"
)
print("OK: przywrocenie najlepszego stanu dziala poprawnie.")
print(f"Dokladnosc walidacyjna po przywroceniu: {accuracy(es_model, X_val, y_val):.3f}")


## 6. Kontrolowane porównanie i finalna ewaluacja — Twoje zadanie

Wybierz **jedną** konfigurację o najlepszej stracie walidacyjnej spośród
wszystkich przetestowanych (sekcje 3–5). Wytrenuj ją (jeśli trzeba,
ponownie) i oceń raz, na zbiorze testowym.


In [ ]:
# TODO:
# 1. zbierz strate walidacyjna KAZDEJ przetestowanej konfiguracji razem z
#    JEJ WYTRENOWANYM MODELEM: baseline (baseline_model, baseline_vl[-1]),
#    kazdy wd i dropout p z wd_results/dropout_results (ktore w sekcji 4
#    powinny juz przechowywac model), early stopping (es_model,
#    es_vl[es_best_epoch] - to jego najlepsza epoka, nie ostatnia)
# 2. wybierz konfiguracje o najnizszej stracie walidacyjnej - UZYJ JUZ
#    WYTRENOWANEGO MODELU dla niej (nie trenuj od nowa - unikasz w ten
#    sposob przypadkowej zmiany wyniku przez inna losowosc)
# 3. policz strate i dokladnosc tego modelu na X_test/y_test
#    (nn.BCEWithLogitsLoss(), accuracy(...)) - to jedyne miejsce w
#    notebooku, gdzie uzywasz X_test
raise NotImplementedError("TODO: finalna ewaluacja")


## 7. Wnioski — Twoje zadanie

Odpowiedz (Markdown, uzasadnione rzeczywistymi liczbami):

1. Która metoda regularyzacji dała największą poprawę względem
   baseline'u? Poprawiła dokładność, stratę, czy obie miary?
2. Czy błąd normalizacji z sekcji 1 dał w Twoim przebiegu duży efekt na
   wynik końcowy, czy mały? Niezależnie od odpowiedzi — dlaczego sama
   procedura jest błędna, nawet jeśli efekt okazał się mały?
3. Jakie jest co najmniej jedno ograniczenie tego eksperymentu, które
   sprawia, że nie powinieneś/aś uogólniać wniosków na "zawsze najlepsza
   jest metoda X"?


_(Twoja odpowiedź tutaj.)_